# Embedding Pipeline Notebook

Notebook này tự động hóa pipeline từ data ingestion -> chunking -> compare strategy -> embedding -> thống kê embedding.

Thiết kế ưu tiên chạy CPU: bắt đầu bằng subset nhỏ qua `ARTICLE_LIMIT`, quan sát tiến độ embed bằng `tqdm` từ CLI, sau đó mới tăng limit hoặc chạy full corpus.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)

## 1. Cấu hình

- `RAW_CSV_PATH`: CSV đầu vào cho `src.data_ingestion.cli`.
- `ARTICLE_LIMIT`: số article đưa vào chunking/embedding. Đặt `None` để chạy full corpus.
- `STRATEGIES`: các chiến thuật chunking cần embed.
- `BATCH_SIZE`: CPU nên bắt đầu thấp, ví dụ 4 hoặc 8.

In [ ]:
# Chọn một CSV đang có trong repo. Có thể đổi sang validation_new.csv/test_new.csv hoặc CSV tổng của bạn.
RAW_CSV_PATH = PROJECT_ROOT / "Dataset" / "Create_QA_Vietonline" / "VietOnlineNews" / "train_new.csv"

PROCESSED_JSONL = PROJECT_ROOT / "src" / "embed" / "output" / "data" / "vieonline_news_clean.jsonl"
REVIEW_JSONL = PROJECT_ROOT / "src" / "embed" / "output" / "data" / "vieonline_news_human_review.jsonl"
CHUNK_OUTPUT_DIR = PROJECT_ROOT / "src" / "embed" / "output" / "chunks"
DENSE_OUTPUT_ROOT = PROJECT_ROOT / "src" / "embed" / "output" / "dense"
SAMPLE_REPORT = PROJECT_ROOT / "src" / "embed" / "output" / "embedding_strategy_samples.md"

# CPU-friendly default. Đặt None để chạy toàn bộ file sau khi đã smoke test.
ARTICLE_LIMIT = 200
STRATEGIES = ["token", "llamaindex", "structured"]  # thêm "langchain_recursive" nếu muốn so sánh đủ 4 strategy

CHUNK_SIZE = 400
OVERLAP = 80
SMALL_ARTICLE_CHARS = 1000
BATCH_SIZE = 4
MODEL_NAME = "intfloat/multilingual-e5-large"
SAMPLE_SIZE = 5

for path in [PROCESSED_JSONL.parent, CHUNK_OUTPUT_DIR, DENSE_OUTPUT_ROOT, SAMPLE_REPORT.parent]:
    path.mkdir(parents=True, exist_ok=True)

config = {
    "raw_csv": str(RAW_CSV_PATH),
    "processed_jsonl": str(PROCESSED_JSONL),
    "chunk_output_dir": str(CHUNK_OUTPUT_DIR),
    "dense_output_root": str(DENSE_OUTPUT_ROOT),
    "article_limit": ARTICLE_LIMIT,
    "strategies": STRATEGIES,
    "chunk_size": CHUNK_SIZE,
    "overlap": OVERLAP,
    "batch_size": BATCH_SIZE,
    "model_name": MODEL_NAME,
}
display(config)

## 2. Helper chạy CLI có streaming output

Các lệnh bên dưới dùng `sys.executable -m ...` để chạy đúng Python environment của notebook. Output được stream trực tiếp để thấy progress bar của chunking và embedding.

In [ ]:
def run_cli(args, cwd=PROJECT_ROOT):
    args = [str(item) for item in args]
    printable = " ".join(args)
    print(f"\n$ {printable}\n")
    started = time.perf_counter()
    process = subprocess.Popen(
        args,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        lines.append(line)
    return_code = process.wait()
    elapsed = time.perf_counter() - started
    print(f"\n[exit={return_code}] elapsed={elapsed:.2f}s")
    if return_code != 0:
        raise RuntimeError(f"Command failed: {printable}")
    return "".join(lines)

## 3. Data ingestion

In [ ]:
ingest_cmd = [
    sys.executable, "-m", "src.data_ingestion.cli",
    "--input", RAW_CSV_PATH,
    "--output", PROCESSED_JSONL,
    "--review-output", REVIEW_JSONL,
]
run_cli(ingest_cmd)

## 4. Chunking theo strategy

Bước này dùng CLI `src.chunking.cli`. Nếu chạy CPU, giữ `ARTICLE_LIMIT` nhỏ trước để giảm thời gian embed sau đó.

In [ ]:
chunk_cmd = [
    sys.executable, "-m", "src.chunking.cli",
    "--input", PROCESSED_JSONL,
    "--output-dir", CHUNK_OUTPUT_DIR,
    "--strategies", *STRATEGIES,
    "--chunk-size", CHUNK_SIZE,
    "--overlap", OVERLAP,
    "--small-article-chars", SMALL_ARTICLE_CHARS,
]
if ARTICLE_LIMIT is not None:
    chunk_cmd += ["--limit", ARTICLE_LIMIT]
run_cli(chunk_cmd)

## 5. Xem mẫu khác biệt input embedding giữa các strategy

In [ ]:
chunk_files = [CHUNK_OUTPUT_DIR / f"vieonline_news_chunks_{strategy}.jsonl" for strategy in STRATEGIES]
compare_cmd = [
    sys.executable, "-m", "src.embed.compare_embedding_inputs",
    "--inputs", *chunk_files,
    "--sample-articles", 3,
    "--output", SAMPLE_REPORT,
]
run_cli(compare_cmd)
display(Markdown(SAMPLE_REPORT.read_text(encoding="utf-8")))

## 6. Embed từng strategy và quan sát tiến độ

Cell này là bước chậm nhất trên CPU. `tqdm` sẽ hiển thị tiến độ theo batch. Nếu quá chậm, giảm `ARTICLE_LIMIT`, giảm số `STRATEGIES`, hoặc giữ `BATCH_SIZE=4`.

In [ ]:
for strategy, chunk_file in zip(STRATEGIES, chunk_files):
    output_dir = DENSE_OUTPUT_ROOT / strategy
    embed_cmd = [
        sys.executable, "-m", "src.embed.embed_chunks",
        "--input", chunk_file,
        "--output-dir", output_dir,
        "--model", MODEL_NAME,
        "--batch-size", BATCH_SIZE,
        "--sample-size", SAMPLE_SIZE,
        "--show-samples",
    ]
    run_cli(embed_cmd)

## 7. Bảng thống kê embedding

In [ ]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

rows = []
for strategy in STRATEGIES:
    strategy_dir = DENSE_OUTPUT_ROOT / strategy
    stats_path = strategy_dir / "embedding_stats.json"
    manifest_path = strategy_dir / "manifest.json"
    if not stats_path.exists():
        print(f"Missing stats for {strategy}: {stats_path}")
        continue
    stats = load_json(stats_path)
    manifest = load_json(manifest_path)
    rows.append({
        "strategy": strategy,
        "chunks": stats["chunks_embedded"],
        "dimension": stats["embedding_dimension"],
        "batch_size": stats["batch_size"],
        "elapsed_seconds": stats["elapsed_seconds"],
        "chunks_per_second": stats["chunks_per_second"],
        "avg_tokens": stats["token_stats"]["avg"],
        "max_tokens": stats["token_stats"]["max"],
        "avg_chars": stats["char_stats"]["avg"],
        "max_chars": stats["char_stats"]["max"],
        "embeddings_path": manifest["embeddings_path"],
        "stats_path": str(stats_path),
    })

embed_stats_df = pd.DataFrame(rows).sort_values("strategy")
display(embed_stats_df)

summary_csv = PROJECT_ROOT / "src" / "embed" / "output" / "embedding_stats_summary.csv"
embed_stats_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
print("Saved:", summary_csv)

## 8. Debug chunk dài nhất và mẫu đã embed

In [ ]:
for strategy in STRATEGIES:
    strategy_dir = DENSE_OUTPUT_ROOT / strategy
    stats_path = strategy_dir / "embedding_stats.json"
    samples_path = strategy_dir / "debug_samples.jsonl"
    if not stats_path.exists():
        continue
    stats = load_json(stats_path)
    display(Markdown(f"### {strategy}: longest chunks"))
    display(pd.DataFrame(stats["longest_chunks"]))
    if samples_path.exists():
        samples = [json.loads(line) for line in samples_path.read_text(encoding="utf-8").splitlines() if line.strip()]
        display(Markdown(f"### {strategy}: debug samples"))
        display(pd.DataFrame(samples))

## 9. Search thử trên một strategy đã embed

In [ ]:
QUERY = "Tin tức về công nghệ AI tại Việt Nam"
SEARCH_STRATEGY = STRATEGIES[0]

search_cmd = [
    sys.executable, "-m", "src.embed.dense_search",
    "--index-dir", DENSE_OUTPUT_ROOT / SEARCH_STRATEGY,
    "--query", QUERY,
    "--top-k", 5,
    "--model", MODEL_NAME,
]
run_cli(search_cmd)

## 10. Eval retrieval trên 50 QA mẫu

Cell này đánh giá các dense index đã embed bằng qrels ở mức `article_id`. Mỗi retrieved chunk được xem là đúng nếu `article_id` của chunk nằm trong `source_article_ids` hoặc `article_id` của QA.

Metric chính để xếp hạng là `nDCG@10`. Các metric khác dùng để giải thích trade-off chất lượng/chi phí.

In [ ]:
QA_PATH = PROJECT_ROOT / "Dataset" / "QA_Claude" / "QA_output.csv"
QA_SAMPLE_SIZE = 50
EVAL_TOP_K = 10

qa_df = pd.read_csv(QA_PATH)
if "is_possible" in qa_df.columns:
    qa_df = qa_df[qa_df["is_possible"].astype(str).str.lower().isin(["true", "1", "yes"])]
qa_df = qa_df.dropna(subset=["question"]).head(QA_SAMPLE_SIZE).copy()

print("QA samples:", len(qa_df))
display(qa_df[[col for col in ["id", "article_id", "question", "source_article_ids", "qa_type"] if col in qa_df.columns]].head())

In [ ]:
import math
import statistics
import numpy as np
from sentence_transformers import SentenceTransformer

from src.embed.embed_chunks import prepare_query_text
from src.embed.dense_search import load_index


def parse_qrels(row):
    values = set()
    for col in ["source_article_ids", "article_id"]:
        if col not in row or pd.isna(row[col]):
            continue
        raw = row[col]
        if isinstance(raw, (list, tuple, set)):
            parts = raw
        else:
            text = str(raw).strip()
            try:
                parsed = json.loads(text)
                parts = parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                parts = text.replace(";", ",").replace("|", ",").split(",")
        for part in parts:
            value = str(part).strip().strip("[]'\"")
            if value and value.lower() not in {"nan", "none", ""}:
                values.add(value)
    return values


def dcg_at_k(relevances, k):
    return sum(rel / math.log2(idx + 2) for idx, rel in enumerate(relevances[:k]))


def metrics_for_results(results, relevant_article_ids, k=10):
    retrieved_article_ids = [str(item.get("article_id")) for item in results[:k]]
    relevances = [1 if article_id in relevant_article_ids else 0 for article_id in retrieved_article_ids]
    ideal_relevances = [1] * min(len(relevant_article_ids), k)
    ideal_dcg = dcg_at_k(ideal_relevances, k)
    ndcg = dcg_at_k(relevances, k) / ideal_dcg if ideal_dcg else 0.0

    def recall_at(cutoff):
        found = set(retrieved_article_ids[:cutoff]) & relevant_article_ids
        return len(found) / len(relevant_article_ids) if relevant_article_ids else 0.0

    def hit_at(cutoff):
        return 1.0 if set(retrieved_article_ids[:cutoff]) & relevant_article_ids else 0.0

    mrr = 0.0
    for idx, article_id in enumerate(retrieved_article_ids[:k], start=1):
        if article_id in relevant_article_ids:
            mrr = 1.0 / idx
            break

    return {
        "nDCG@10": ndcg,
        "Recall@5": recall_at(5),
        "Recall@10": recall_at(10),
        "MRR@10": mrr,
        "Hit@1": hit_at(1),
        "Hit@5": hit_at(5),
    }


def percentile(values, pct):
    if not values:
        return 0.0
    values = sorted(values)
    index = min(len(values) - 1, math.ceil((pct / 100) * len(values)) - 1)
    return values[index]


def directory_size_mb(path):
    path = Path(path)
    return sum(file.stat().st_size for file in path.rglob("*") if file.is_file()) / (1024 * 1024)


def search_arrays(query, embeddings, metadata, model, top_k):
    query_vector = model.encode([prepare_query_text(query)], normalize_embeddings=True, show_progress_bar=False)
    query_array = np.asarray(query_vector, dtype=np.float32)[0]
    scores = embeddings @ query_array
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [metadata[int(index)] | {"score": float(scores[int(index)])} for index in top_indices]

In [ ]:
# Load model một lần để query evaluation không tải lại model theo từng query.
query_model = SentenceTransformer(MODEL_NAME)

eval_rows = []
per_query_rows = []

for strategy in STRATEGIES:
    index_dir = DENSE_OUTPUT_ROOT / strategy
    stats_path = index_dir / "embedding_stats.json"
    manifest_path = index_dir / "manifest.json"
    if not (index_dir / "embeddings.npy").exists():
        print(f"Skip {strategy}: missing embeddings.npy")
        continue

    embeddings, metadata = load_index(index_dir)
    stats = load_json(stats_path)
    manifest = load_json(manifest_path)
    query_latencies_ms = []
    metric_rows = []

    for _, qa in qa_df.iterrows():
        relevant_article_ids = parse_qrels(qa)
        if not relevant_article_ids:
            continue
        started = time.perf_counter()
        results = search_arrays(str(qa["question"]), embeddings, metadata, query_model, EVAL_TOP_K)
        latency_ms = (time.perf_counter() - started) * 1000
        query_latencies_ms.append(latency_ms)

        metrics = metrics_for_results(results, relevant_article_ids, k=EVAL_TOP_K)
        metric_rows.append(metrics)
        per_query_rows.append({
            "strategy": strategy,
            "qa_id": qa.get("id"),
            "qa_type": qa.get("qa_type"),
            "question": qa.get("question"),
            "relevant_article_ids": sorted(relevant_article_ids),
            "top_article_ids": [item.get("article_id") for item in results[:EVAL_TOP_K]],
            "latency_ms": latency_ms,
            **metrics,
        })

    if not metric_rows:
        continue

    aggregate = {key: float(np.mean([row[key] for row in metric_rows])) for key in metric_rows[0]}
    eval_rows.append({
        "Model": MODEL_NAME,
        "Chunking": strategy,
        "nDCG@10": aggregate["nDCG@10"],
        "Recall@5": aggregate["Recall@5"],
        "Recall@10": aggregate["Recall@10"],
        "MRR@10": aggregate["MRR@10"],
        "Hit@1": aggregate["Hit@1"],
        "Hit@5": aggregate["Hit@5"],
        "embedding_time_seconds": stats["elapsed_seconds"],
        "chunks_per_second": stats["chunks_per_second"],
        "query_latency_ms_avg": statistics.mean(query_latencies_ms),
        "query_latency_ms_p95": percentile(query_latencies_ms, 95),
        "index_size_mb": directory_size_mb(index_dir),
        "embedding_dimension": manifest["embedding_dimension"],
        "num_chunks": stats["chunks_embedded"],
        "avg_chunk_tokens": stats["token_stats"]["avg"],
        "queries": len(metric_rows),
    })

eval_df = pd.DataFrame(eval_rows)
if not eval_df.empty:
    eval_df = eval_df.sort_values(
        ["nDCG@10", "Recall@10", "query_latency_ms_avg"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    eval_df.insert(0, "Rank", range(1, len(eval_df) + 1))

display(eval_df)

per_query_df = pd.DataFrame(per_query_rows)
eval_output_dir = PROJECT_ROOT / "src" / "embed" / "output" / "eval"
eval_output_dir.mkdir(parents=True, exist_ok=True)
eval_df.to_csv(eval_output_dir / "leaderboard_qa50.csv", index=False, encoding="utf-8-sig")
per_query_df.to_json(eval_output_dir / "per_query_results_qa50.jsonl", orient="records", lines=True, force_ascii=False)
print("Saved leaderboard:", eval_output_dir / "leaderboard_qa50.csv")
print("Saved per-query results:", eval_output_dir / "per_query_results_qa50.jsonl")

## 11. Notes đánh giá

Quy tắc đọc bảng:

- Metric xếp hạng chính: `nDCG@10`.
- Nếu `nDCG@10` gần nhau, ưu tiên `Recall@10` cao hơn và `query_latency_ms_avg` thấp hơn.
- Không so sánh các dòng nếu dùng khác query set hoặc khác qrels.
- Với E5, notebook dùng prefix `query: ` cho câu hỏi và `passage: ` cho chunk trong bước embed.
- Kết quả này là eval nhanh trên khoảng 50 QA mẫu, chưa thay thế evaluation đầy đủ cho báo cáo cuối.

Các câu hỏi cần trả lời từ leaderboard:

1. Strategy chunking nào ổn định nhất?
2. Model/chunking nào đạt retrieval tốt nhất theo `nDCG@10`?
3. Cấu hình nào trade-off tốt giữa chất lượng, latency và index size?
4. Cấu hình nào nên dùng cho pipeline RAG cuối cùng?